# BO7 Enemy Detector — Train on Colab, Export ONNX

Flow:
1. Verify GPU (Runtime → Change runtime type → **T4 GPU**).
2. Install `ultralytics` (+ `roboflow` for Option A).
3. Get the dataset — **either** from Roboflow (free dataset download) **or** from a ZIP you upload.
4. Train YOLOv11n / YOLOv8n.
5. Export `best.pt` → `best.onnx`.
6. Download both to your Mac.

Roboflow's free tier is used **only to download the dataset** — training and weights export happen here in Colab, so no paid Roboflow product is touched.

## 0. GPU check

In [ ]:
!nvidia-smi || echo 'No GPU. Runtime → Change runtime type → T4 GPU, then re-run.'

## 1. Install Ultralytics (+ roboflow)

In [ ]:
!pip install -q -U ultralytics roboflow

## 2. Get the dataset — pick ONE option

### Option A — Pull straight from Roboflow (recommended; free)

On Roboflow: **Versions → your version → Download Dataset → format `YOLOv8` → Show download code**. Copy the values from their snippet into the cell below, then run it. (API key is tied to your account; free.)

In [ ]:
# ---- CONFIG (Option A) ----
# Paste the api_key from Roboflow's "Show download code" snippet.
# The rest matches your forked project.
ROBOFLOW_API_KEY = 'PASTE_KEY_HERE'
RF_WORKSPACE    = 'bobos-workspace-k56b2'
RF_PROJECT      = 'bo7-du6xg-bnnfh'
RF_VERSION      = 1
DATA_FORMAT     = 'yolov8'   # or 'yolov11'

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
dataset = project.version(RF_VERSION).download(DATA_FORMAT)

DATA_YAML = f'{dataset.location}/data.yaml'
print('DATA_YAML =', DATA_YAML)
!head -50 {DATA_YAML}

### Option B — Upload your own labeled ZIP

Only run this if you're skipping Option A. ZIP layout expected:

```
mydata.zip
├── images/{train,val}/*.jpg
├── labels/{train,val}/*.txt
└── data.yaml   (names: {0: Enemy})
```

In [ ]:
from google.colab import files
import zipfile, os, glob

uploaded = files.upload()
zip_name = next(iter(uploaded))
os.makedirs('/content/mydata', exist_ok=True)
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall('/content/mydata')

candidates = glob.glob('/content/mydata/**/data.yaml', recursive=True)
assert candidates, 'No data.yaml found inside the zip.'
DATA_YAML = candidates[0]
print('DATA_YAML =', DATA_YAML)
!head -50 {DATA_YAML}

## 3. Train

Defaults: YOLOv11n @ 640px, 80 epochs, batch 16, T4 GPU. Tweak if you like:
- `yolov8n.pt` — known-good with your `yolo_live.py` postprocess.
- `yolo11n.pt` — same head format, newer, slightly better.
- `yolov8s.pt` / `yolo11s.pt` — small (better accuracy, ~2× slower).

In [ ]:
MODEL    = 'yolo11n.pt'
IMGSZ    = 640
EPOCHS   = 80
BATCH    = 16
PATIENCE = 20
RUN_NAME = 'bo7_enemy'

!yolo train model={MODEL} data={DATA_YAML} imgsz={IMGSZ} epochs={EPOCHS} batch={BATCH} patience={PATIENCE} name={RUN_NAME} device=0 plots=True

## 4. Results

In [ ]:
import glob, os
run_dir = sorted(glob.glob('runs/detect/' + RUN_NAME + '*'))[-1]
print('Run dir:', run_dir)
!ls -la {run_dir}/weights
!tail -n 5 {run_dir}/results.csv 2>/dev/null || echo '(no results.csv)'

from IPython.display import Image, display
for p in ['results.png', 'confusion_matrix.png', 'PR_curve.png']:
    f = os.path.join(run_dir, p)
    if os.path.exists(f):
        print(p); display(Image(f))

## 5. Export ONNX

In [ ]:
BEST_PT = f'{run_dir}/weights/best.pt'
!yolo export model={BEST_PT} format=onnx imgsz={IMGSZ} opset=12 dynamic=False simplify=True
BEST_ONNX = BEST_PT.replace('.pt', '.onnx')
!ls -la {BEST_ONNX}

## 6. Download `best.pt` + `best.onnx`

In [ ]:
from google.colab import files
files.download(BEST_PT)
files.download(BEST_ONNX)

## 7. Use the ONNX locally

Drop `best.onnx` into `/Users/stunner/yolo/` and run:

```bash
python yolo_live.py --model best.onnx --data data.yaml --aimbot --classes Enemy
```

For faster CPU FPS: train at `imgsz=320` and run with `--input-size 320`.